In [7]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.data_selection.data_selector import Data_selector

In [8]:
import pandas as pd
folder_path = os.path.join(project_root, "data", "processed", "integrated.csv")
df_zero = pd.read_csv(folder_path)

In [9]:
folder_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(folder_path)

In [10]:
data_set = {"name":[],"code":[],"gi":[],"g0":[],"g1":[],"g2":[],"g3":[],"g4":[],"g5":[],"n_T":[],"n_G":[],"n_S":[]}
ds = Data_selector(df)
ds_zero = Data_selector(df_zero)
for _,row in df_zero[["name","code"]].drop_duplicates().iterrows():
    name = row["name"]
    code = row["code"]
    data_set["name"].append(name)
    data_set["code"].append(code)
    unit_ds = Data_selector(ds.filter_name_code(name,code))
    
    df_zero_filtered = ds_zero.filter_name_code(name,code)
    ll = len(df_zero_filtered)
    data_set["gi"].append(ll)
    data_set["n_T"].append(int(100*df_zero_filtered['temperature'].isnull().sum()/ll))
    data_set["n_S"].append(int(100*(df_zero_filtered['scadaf'].isnull() | df_zero_filtered['temp_sens'].isnull()).sum()/ll))
    data_set["n_G"].append(int(100*df_zero_filtered['generation'].isnull().sum()/ll))
    
    for goodness in range(0,6):
        gx = f"g{goodness}"
        l = len(unit_ds.select_peaks(goodness))
        data_set[gx].append(int((l/ll)*100))
        

In [14]:
df_data = pd.DataFrame(data_set)

In [15]:
df_data["H1"] = df_data["n_T"] + df_data["g0"]

In [16]:
df_data[df_data["code"].str.startswith("G")]

,name,code,gi,g0,g1,g2,g3,g4,g5,n_T,n_G,n_S,H1
0,پرند,G13,12480,91,35,16,16,16,13,0,0,10,91
2,پرند,G15,12480,91,34,16,16,16,12,0,0,10,91
3,پرند,G14,12480,91,35,15,15,15,12,0,0,10,91
5,پرند,G12,12480,91,34,21,8,8,7,0,0,10,91
6,پرند,G11,35208,70,20,11,7,7,5,0,1,28,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,حافظ,G13,12480,47,16,9,9,9,6,0,0,53,47
102,عسلویه,G11,35208,82,26,18,12,12,5,0,1,16,82
103,حافظ,G12,12480,47,17,9,2,2,1,0,0,53,47
104,سیکل ترکیبی شیروان,G12,12480,94,35,14,14,14,4,0,0,20,94


In [17]:
df_data

,name,code,gi,g0,g1,g2,g3,g4,g5,n_T,n_G,n_S,H1
0,پرند,G13,12480,91,35,16,16,16,13,0,0,10,91
1,پرند,S3,12480,0,0,0,0,0,0,0,0,10,0
2,پرند,G15,12480,91,34,16,16,16,12,0,0,10,91
3,پرند,G14,12480,91,35,15,15,15,12,0,0,10,91
4,پرند,S1,12480,0,0,0,0,0,0,0,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,حافظ,G12,12480,47,17,9,2,2,1,0,0,53,47
104,سیکل ترکیبی شیروان,G12,12480,94,35,14,14,14,4,0,0,20,94
105,سیکل ترکیبی شیروان,S3,6504,0,0,0,0,0,0,0,0,12,0
106,حافظ,G11,35208,50,16,8,8,8,4,0,1,49,50


In [18]:
df_bad = df_data[df_data["H1"]<95]

In [19]:
df_bad

,name,code,gi,g0,g1,g2,g3,g4,g5,n_T,n_G,n_S,H1
0,پرند,G13,12480,91,35,16,16,16,13,0,0,10,91
1,پرند,S3,12480,0,0,0,0,0,0,0,0,10,0
2,پرند,G15,12480,91,34,16,16,16,12,0,0,10,91
3,پرند,G14,12480,91,35,15,15,15,12,0,0,10,91
4,پرند,S1,12480,0,0,0,0,0,0,0,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,حافظ,G12,12480,47,17,9,2,2,1,0,0,53,47
104,سیکل ترکیبی شیروان,G12,12480,94,35,14,14,14,4,0,0,20,94
105,سیکل ترکیبی شیروان,S3,6504,0,0,0,0,0,0,0,0,12,0
106,حافظ,G11,35208,50,16,8,8,8,4,0,1,49,50


In [20]:
 df_data["n_T"]+df_data["g0"]

0      91
1       0
2      91
3      91
4       0
       ..
103    47
104    94
105     0
106    50
107     0
Length: 108, dtype: int64

In [21]:
data_set = {"name":[],"code":[],"gi":[],"g0":[],"g1":[],"g2":[],"g3":[],"g4":[],"g5":[],"n_T":[],"n_G":[]}
ds = Data_selector(df_bad)
for _,row in df_bad[["name","code"]].drop_duplicates().iterrows():
    name = row["name"]
    code = row["code"]

In [65]:
def get_null(df, df_name_code):
    infos = []
    for _, row in df_name_code.iterrows():
        name = row["name"]
        code = row["code"]
        unit = Data_selector(df).filter_name_code(name, code)
        g = (unit.isnull().sum() / len(unit)).astype(object)
        g.loc["name"] = name
        g.loc["code"] = code
        infos.append(g)
    df_infos = pd.DataFrame(infos)

    drop_cols = []
    for col in df_infos.columns:
        try:
            m = sum(df_infos[col])
            if m == 0:
                print(col)
                drop_cols.append(col)
        except:
            pass
    df_infos2 = df_infos.drop(columns=drop_cols)
    df_infos2.columns[(df_infos2 == 1).all()]

    return df_infos2

In [67]:
get_null(df_zero, df_bad[["name","code"]])

id
date
hour
load_level
forecasted_load
required
status
evapotranspiration


,name,code,declared,generation,temp_sens,scadaf,temperature,humidity,dew,apparent_temperature,precipitation,rain,snow,surface_pressure,wind_speed,wind_direction
0,شهدای پاکدشت - دماوند,G16,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
1,شهدای پاکدشت - دماوند,G11,0.000682,0.023177,0.296154,0.093388,0.001449,0.001449,0.001449,0.001449,0.001477,0.001477,0.001477,0.001449,0.001449,0.001449
2,شهدای پاکدشت - دماوند,G13,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
3,شهدای پاکدشت - دماوند,G17,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
4,شهدای پاکدشت - دماوند,G14,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
5,شهدای پاکدشت - دماوند,G21,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
6,شهدای پاکدشت - دماوند,G20,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
7,شهدای پاکدشت - دماوند,G15,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
8,شهدای پاکدشت - دماوند,G22,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
9,شهدای پاکدشت - دماوند,G12,0.000000,0.001923,0.131330,0.252965,0.004087,0.004087,0.004087,0.004087,0.004167,0.004167,0.004167,0.004087,0.004087,0.004087
